# Genixcure · PacBio HiFi 단일 Plasmid 원형 정렬 파일럿

**목적**: 기존 `PLA_6_13_hifi_unique_sequences.csv.gz` (`sequence,read_count,length_bp`)를 분석하여 원형 DNA의 시작 위치·방향 차이 때문에 분리된 서열을 **단일 plasmid reference에 매핑해 그룹화**합니다. 동일 플라스미드 1종은 **검증할 가설**이지 전제된 사실이 아닙니다.

**원본**: [PacBio 결과 폴더](https://drive.google.com/drive/folders/12IFMSn8buE2A4GN0TC68Yv3cDl3abRCV) · [Genixcure 원본](https://drive.google.com/drive/folders/1vv7WLOiYTZ0FGXPmFciIN6d9sM6Rhg58)

1. 기존 9.66GB 압축 CSV를 한 번 순차적으로 읽으면서 전체 Count 분포와 read 길이 분포를 계산하고, 샘플용 FASTA(원본 read_count로 가중 추출)를 만듭니다. **122.7GB BAM 재다운로드 불필요**.
2. 가능하면 **검증된 플라스미드 reference FASTA** 사용. 없으면 샘플 HiFi read로 Flye *탐색적* de novo assembly를 시도합니다. Flye가 원형임을 표시한 단일 contig를 얻지 못하면 멈추며, 임의의 read를 플라스미드라 단정하지 않습니다.
3. Minimap2로 `reference + reference` (2회 연결)에 정렬하여 원형 경계 횡단과 역방향 read를 허용합니다. 정렬 위치는 `start % plasmid_length`로 정규화합니다.
4. **잠정적인 같은 플라스미드 read**, 부분 정렬, concatemer 후보, 미배정 read를 구분하고, 가중 read count로 요약합니다. **99.5% identity는 파일럿 분류 기준**이며 최종 변이 확정 기준이 아닙니다.

⚠️ **서로 다른 plasmid variant를 하나로 지우는 작업이 아닙니다.** 원형 정렬·coverage·변이체/concatemer 검토가 필요합니다. 파일럿을 통과한 다음 `RUN_FULL_DATASET=True`로 전체 고유 서열에 같은 기준을 적용할 수 있지만, 9.66GB 재읽기와 수백만 서열 매핑에 많은 시간이 걸립니다. 기본값은 파일럿만 진행합니다.


In [ ]:
!command -v minimap2 >/dev/null || (apt-get update -qq >/dev/null && apt-get install -y -qq minimap2 >/dev/null)
!command -v flye >/dev/null || pip -q install flye
!minimap2 --version
!flye --version


In [ ]:
from google.colab import drive
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import gzip,csv,json,random,heapq,math,subprocess,time,shutil,os
import pandas as pd
import matplotlib.pyplot as plt

drive.mount('/content/drive')
SOURCE=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/PacBio_PLA_6_13/PLA_6_13_hifi_unique_sequences.csv.gz')
OUT=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/PacBio_PLA_6_13/Circular_Pilot')
WORK=Path('/content/plasmid_circular_pilot')
OUT.mkdir(parents=True,exist_ok=True); WORK.mkdir(parents=True,exist_ok=True)
assert SOURCE.is_file(), f'파일 없음: {SOURCE}'

# 직접 제공한 원형 플라스미드 FASTA가 있다면 이 경로로 지정 (가장 권장)
REFERENCE_FASTA=None  # 예: '/content/drive/MyDrive/Genixcure/plasmid_reference.fasta'
ASSEMBLE_IF_NO_REFERENCE=True
SAMPLE_UNIQUE=12000              # RAM/시간 고려한 파일럿 가중 샘플 수
SEED=20261103
FLYE_GENOME_SIZE='5k'            # 플라스미드 길이를 알면 변경; Flye 조립 전용 추정치
FLYE_THREADS=4
MIN_IDENTITY=0.995               # 시험 기준: 99.5%
MIN_QUERY_COVERAGE=0.90         # read의 90% 이상 정렬
MIN_REF_COVERAGE=0.85           # full-length 후보를 위한 reference coverage
RUN_FULL_DATASET=False          # 명시적으로 True일 때만 대규모 매핑
FULL_CHUNK_UNIQUE=25000
print('Source:',SOURCE,'size GB=',round(SOURCE.stat().st_size/1e9,2))


## 1. CSV 한 번 읽고 실제 Singleton·중복도 확인, 샘플 추출

Sequence가 4.6백만 줄이므로 전체를 Pandas RAM에 올리지 않습니다. `-log(U)/read_count` 방식의 가중 reservoir에서 **Unique 항목을 비복원 추출**하고 원래 Count를 보존합니다. 이 샘플은 전체 reads의 정확한 랜덤표본이 아니라 **탐색적 가중 Unique 표본**입니다. 파일 내용이 바뀌면 결과도 달라질 수 있으므로 seed와 파일 크기를 기록합니다.


In [ ]:
rng=random.Random(SEED)
length_hist=Counter()            # raw reads로 가중한 read 길이
unique_length_hist=Counter()
count_hist=Counter()
heap=[]
n_unique=0; n_reads=0; n_singletons=0
start=time.monotonic()
with gzip.open(SOURCE,'rt',newline='') as fin:
    reader=csv.DictReader(fin)
    assert reader.fieldnames==['sequence','read_count','length_bp'],reader.fieldnames
    for row in reader:
        seq=row['sequence'].upper(); count=int(row['read_count']); length=int(row['length_bp'])
        if not seq or count<=0 or length!=len(seq):
            raise ValueError('Invalid sequence/count/length row')
        n_unique+=1; n_reads+=count
        n_singletons+=(count==1)
        count_hist[min(count,100)]+=1
        length_hist[length]+=count
        unique_length_hist[length]+=1
        key=-math.log(max(rng.random(),1e-300))/count
        item=(-key,n_unique,seq,count)
        if len(heap)<SAMPLE_UNIQUE: heapq.heappush(heap,item)
        elif item>heap[0]: heapq.heapreplace(heap,item)
        if n_unique%500000==0: print(f'Scanned {n_unique:,} unique rows ({(time.monotonic()-start)/60:.1f} min)',flush=True)

sample=sorted(heap,reverse=True)
SAMPLE_FASTA=WORK/'weighted_pilot_reads.fasta'
SAMPLE_META=WORK/'sample_metadata.csv'
with open(SAMPLE_FASTA,'w') as f, open(SAMPLE_META,'w',newline='') as m:
    wr=csv.writer(m);wr.writerow(['query_id','read_count','length_bp'])
    for i,(_,_,seq,count) in enumerate(sample):
        name=f'query_{i:07d}'
        f.write(f'>{name} count={count}\n{seq}\n')
        wr.writerow([name,count,len(seq)])
summary={'all_unique_sequences':n_unique,'all_hifi_reads':n_reads,
         'singleton_unique_sequences':n_singletons,
         'singleton_fraction_of_unique':n_singletons/max(n_unique,1),
         'singleton_fraction_of_reads':n_singletons/max(n_reads,1),
         'pilot_unique_sequences':len(sample),
         'pilot_weighted_reads':sum(it[3] for it in sample),
         'seconds_scanning':round(time.monotonic()-start,2)}
(OUT/'read_count_distribution_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))

sizes=pd.DataFrame(sorted(length_hist.items()),columns=['length_bp','weighted_reads'])
sizes.to_csv(OUT/'all_hifi_length_distribution.csv',index=False)
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(sizes['length_bp'],sizes['weighted_reads'])
ax.set(xlabel='HiFi read length (bp)',ylabel='Reads weighted by count',title='HiFi read length distribution')
ax.grid(alpha=.25)
fig.tight_layout();fig.savefig(OUT/'all_hifi_length_distribution.png',dpi=160);plt.show()


## 2. 대표 원형 플라스미드 reference 확보

가능하면 원 플라스미드 FASTA를 입력하세요. 없다면 Flye로 **탐색적** 조립 후 `assembly_info.txt`에서 circular 표시가 있는 단일 contig만 사용합니다. Flye 조립 실패나 circular 미확인 시 자동으로 단일 플라스미드로 합치지 않습니다. `FLYE_GENOME_SIZE`는 알려진 길이에 맞게 수정하세요.


In [ ]:
def load_fasta_one(path):
    records=[]; name=None; sequence=[]
    with open(path) as fh:
        for line in fh:
            line=line.strip()
            if not line: continue
            if line.startswith('>'):
                if name is not None: records.append((name,''.join(sequence)))
                name=line[1:].split()[0]; sequence=[]
            else: sequence.append(line.upper())
    if name is not None: records.append((name,''.join(sequence)))
    return records

if REFERENCE_FASTA:
    records=load_fasta_one(REFERENCE_FASTA)
    assert len(records)==1,'단일 플라스미드 reference FASTA에는 sequence 하나만 있어야 합니다.'
    REF_SOURCE='user_reference_circularity_not_independently_verified'
else:
    assert ASSEMBLE_IF_NO_REFERENCE,'REFERENCE_FASTA 또는 ASSEMBLE_IF_NO_REFERENCE 설정 필요'
    assembly_dir=WORK/'flye_assembly'
    cmd=['flye','--pacbio-hifi',str(SAMPLE_FASTA),'--genome-size',FLYE_GENOME_SIZE,
         '--out-dir',str(assembly_dir),'--threads',str(FLYE_THREADS),'--plasmids']
    print('Running exploratory Flye assembly:', ' '.join(cmd),flush=True)
    logfile=WORK/'flye_run.log'
    with open(logfile,'w') as log:
        run=subprocess.run(cmd,stdout=log,stderr=subprocess.STDOUT)
    if run.returncode:
        raise RuntimeError('Flye failed. Check '+str(logfile)+'; do not force clustering.')
    info=assembly_dir/'assembly_info.txt'
    assembled=assembly_dir/'assembly.fasta'
    assert assembled.is_file() and info.is_file(),'Flye did not produce contigs/assembly_info.txt'
    records_all=dict(load_fasta_one(assembled))
    circ=[]
    with info.open() as f:
        headers=None
        for line in f:
            if line.startswith('#seq_name'): headers=line.lstrip('#').split(); continue
            if line.startswith('#') or not line.strip(): continue
            parts=line.split()
            if headers and len(parts)>=len(headers):
                r=dict(zip(headers,parts))
                if r.get('circ.','').upper()=='Y' and r.get('seq_name') in records_all:
                    circ.append((r['seq_name'],records_all[r['seq_name']]))
    if len(circ)!=1:
        raise RuntimeError(f'Expected ONE Flye circular contig, found {len(circ)}. Review assembly_info.txt and supply verified REFERENCE_FASTA.')
    records=circ
    REF_SOURCE='flye_single_circular_contig_exploratory'

refname,refseq=records[0]
assert len(refseq)>=500 and set(refseq)<=set('ACGTN'), 'Reference appears invalid'
REF_LEN=len(refseq)
REF_DOUBLE=WORK/'plasmid_ref_doubled.fasta'
REF_SINGLE=OUT/'plasmid_reference_candidate.fasta'
REF_SINGLE.write_text(f'>plasmid_candidate source={REF_SOURCE}\n{refseq}\n')
REF_DOUBLE.write_text(f'>plasmid_doubled\n{refseq+refseq}\n')
print('Candidate length:',REF_LEN,'Reference source:',REF_SOURCE)


## 3. 원형 reference 두 번 연결 후 minimap2 정렬 (파일럿)

2× reference에 alignment를 수행하면 시작점이 끝을 넘나드는 read를 하나의 연속적 대상에 매핑할 수 있습니다. PAF는 primary alignment만 사용하며, read의 `query coverage`, `identity`, `reference span`, 방향, 원형 시작 위치를 기록합니다. 단일 PAF alignment로 구조변이를 배제할 수 없으므로 임계값은 **파일럿 잠정 판정**에만 사용합니다. <https://github.com/lh3/minimap2>


In [ ]:
PILOT_PAF=WORK/'pilot.paf'
cmd=['minimap2','-x','map-hifi','--secondary=no','-c','-t','4',str(REF_DOUBLE),str(SAMPLE_FASTA)]
with open(PILOT_PAF,'w') as f,open(WORK/'minimap2.log','w') as err:
    subprocess.run(cmd,stdout=f,stderr=err,check=True)
print('Pilot PAF size:',PILOT_PAF.stat().st_size)

def parse_paf(path):
    best={}
    with open(path) as f:
        for line in f:
            fields=line.rstrip('\n').split('\t')
            if len(fields)<12: continue
            q,ql,qs,qe,strand,t,tl,ts,te,matches,alen,mapq=fields[:12]
            ql,qs,qe,tl,ts,te,matches,alen,mapq=map(int,(ql,qs,qe,tl,ts,te,matches,alen,mapq))
            tag={x[:2]:x[5:] for x in fields[12:] if len(x)>=5 and x[2:5]==':Z:'}
            tp=next((x[-1] for x in fields[12:] if x.startswith('tp:A:')),None)
            if tp=='S':continue
            if alen<=0:continue
            r={'query_id':q,'query_length':ql,'query_coverage':(qe-qs)/ql,
               'identity':matches/alen,'reference_span':te-ts,'mapq':mapq,
               'strand':strand,'circular_start_0based':ts%REF_LEN,
               'junction_crossing':bool((ts//REF_LEN)!=(max(ts,te-1)//REF_LEN)),
               'alignment_length':alen,'query_start':qs,'query_end':qe}
            rank=((qe-qs),matches,mapq)
            if q not in best or rank>best[q][0]:best[q]=(rank,r)
    return {k:v[1] for k,v in best.items()}

aln=parse_paf(PILOT_PAF)
meta=pd.read_csv(SAMPLE_META).set_index('query_id')
rows=[]
for query,m in meta.iterrows():
    a=aln.get(query)
    if not a: status='unmapped'; a={}
    elif m.length_bp>REF_LEN*1.5: status='concatemer_or_long_read_candidate'
    elif a['query_coverage']<MIN_QUERY_COVERAGE: status='partial_or_structural_variant_candidate'
    elif a['identity']<MIN_IDENTITY: status='divergent_or_sequencing_error_candidate'
    elif a['reference_span']<REF_LEN*MIN_REF_COVERAGE: status='short_covered_region_only'
    else: status='consistent_with_reference_pilot'
    rows.append({'query_id':query,'read_count':int(m.read_count),'read_length_bp':int(m.length_bp),
                 'classification':status,**a})
result=pd.DataFrame(rows)
result.to_csv(OUT/'pilot_read_assignments.csv',index=False)
grouped=result.groupby('classification',dropna=False).agg(unique_sequences=('query_id','size'),
                                               weighted_reads=('read_count','sum')).reset_index()
grouped.to_csv(OUT/'pilot_plasmid_group_summary.csv',index=False)
display(grouped)
print('Sampled unique:',len(result),'weighted reads:',int(result.read_count.sum()))
print('Caution: pilot read_count weights do not form an unbiased estimate of total fractions.')


## 4. 전체 4.6백만 Unique mapping (선택)

**기본 `RUN_FULL_DATASET=False`**. 진행할 경우 원본 9.66GB CSV를 다시 순차 읽으면서 25,000 Unique씩 FASTA chunk를 만들고 매핑·집계합니다. 메모리는 chunk에 제한되며, 결과는 read별 파일 대신 **분류별 합계 및 진행 checkpoint**로 저장합니다. 중단된 chunk는 다시 시도할 수 있지만, CSV 전체를 다시 읽을 수 있어 시간이 소요됩니다.


In [ ]:
if not RUN_FULL_DATASET:
    print('Full-data analysis skipped (RUN_FULL_DATASET=False).')
else:
    allrows=[]; chunk=[]; chunkid=0; seen=0
    chunk_folder=WORK/'full_chunk'; chunk_folder.mkdir(exist_ok=True)
    checkpoint=OUT/'full_progress.json'
    def process_chunk(chunk,chunkid):
        fa=chunk_folder/f'chunk_{chunkid:05d}.fa'
        paf=chunk_folder/f'chunk_{chunkid:05d}.paf'
        with fa.open('w') as f:
            for q,seq,count in chunk: f.write(f'>{q}\n{seq}\n')
        cmd=['minimap2','-x','map-hifi','--secondary=no','-c','-t','4',str(REF_DOUBLE),str(fa)]
        with paf.open('w') as f,open(chunk_folder/'minimap2.log','a') as err:
            subprocess.run(cmd,stdout=f,stderr=err,check=True)
        mapped=parse_paf(paf)
        counter=Counter()
        for q,seq,count in chunk:
            a=mapped.get(q)
            if not a: status='unmapped'
            elif len(seq)>REF_LEN*1.5: status='concatemer_or_long_read_candidate'
            elif a['query_coverage']<MIN_QUERY_COVERAGE: status='partial_or_structural_variant_candidate'
            elif a['identity']<MIN_IDENTITY: status='divergent_or_sequencing_error_candidate'
            elif a['reference_span']<REF_LEN*MIN_REF_COVERAGE: status='short_covered_region_only'
            else: status='consistent_with_reference_pilot'
            counter[(status,'unique')]+=1
            counter[(status,'reads')]+=count
        fa.unlink(); paf.unlink()
        return counter
    totals=Counter()
    with gzip.open(SOURCE,'rt',newline='') as fin:
        for row in csv.DictReader(fin):
            seen+=1
            chunk.append((f'u{seen}',row['sequence'],int(row['read_count'])))
            if len(chunk)==FULL_CHUNK_UNIQUE:
                totals.update(process_chunk(chunk,chunkid)); chunk=[]; chunkid+=1
                checkpoint.write_text(json.dumps({'chunks_done':chunkid,'unique_scanned':seen,'totals':{f'{a}:{b}':v for (a,b),v in totals.items()}},indent=2))
                print(f'Complete {seen:,} unique; {chunkid} chunks',flush=True)
        if chunk:
            totals.update(process_chunk(chunk,chunkid))
    total_df=pd.DataFrame([{'classification':s,'unique_sequences':totals[(s,'unique')],
                            'weighted_reads':totals[(s,'reads')]}
                            for s in sorted({s for s,_ in totals})])
    assert int(total_df.unique_sequences.sum())==n_unique
    assert int(total_df.weighted_reads.sum())==n_reads
    total_df.to_csv(OUT/'full_plasmid_group_summary.csv',index=False)
    display(total_df)


## 해석 기준

- `consistent_with_reference_pilot` = **참조 플라스미드와 양립 가능한 read 후보**. 동일 분자/동일 변이형 확정이 아님.
- `partial_or_structural_variant_candidate` = 부분 read, 구조변이 또는 alignment 실패 가능성. 임의로 동일 플라스미드로 합치지 않음.
- `concatemer_or_long_read_candidate` = 길이만으로 선별한 후보. 다중 반복 원형 통과 여부를 별도로 확인해야 함.
- 이 파일럿은 **정확한 variant-specific consensus와 read count를 자동 생성하지 않습니다.** 최종 개별 변이형 구분에는 원형 경계, coverage, indel/SNV 재검증이 필요합니다.
- 원시 PacBio `.bam`에서의 `samtools fastq` 방향 처리·CCS 형성과 여기서 분석하는 CSV의 전처리 차이를 주의하세요.
